In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from getdist import plots, loadMCSamples

# Path to your chains
chain_path = './outputs/background'
samples = loadMCSamples(chain_path, settings={'ignore_rows': 0.3})

# In 1.6.4, the sum of weights is stored in 'norm'
print(f"Successfully loaded chains: {samples.getName()}")
print(f"Total weight (norm): {samples.norm:.2f}")
print(f"Number of samples (rows): {samples.numrows}")
print(f"Number of parameters: {samples.n}")

Successfully loaded chains: background
Total weight (norm): 81234.00
Number of samples (rows): 27274
Number of parameters: 27


In [2]:
# # Your LaTeX labels dictionary
# labels_dict = {
#     'H0': r'$H_0$',
#     'omega_b': r'$\omega_b h^2$',
#     'omega_cdm': r'$\omega_{cdm} h^2$',
#     'a_exo': r'$a_{\rm exo}$',
#     'b_exo': r'$b_{\rm exo}$',
#     'Omega_x0': r'$\Omega_{x0}$',
#     'Omega_m': r'$\Omega_m$',
#     'sigma8': r'$\sigma_8$',
#     'S8': r'$S_8$'
# }

# # Accessing the ParamNames object directly to set labels
# for name, label in labels_dict.items():
#     param = samples.paramNames.parWithName(name)
#     if param:
#         param.label = label
#     else:
#         print(f"Warning: Parameter '{name}' not found in chains.")

# print("Labels updated successfully.")

In [3]:
# Create a new samples object that EXCLUDES the problematic parameter entirely
# This is the safest way to prevent singular matrix errors
samples_fixed = samples.copy()
samples_fixed.setParamNames(samples.paramNames) # ensure names carry over
params_to_keep = ['H0', 'Omega_m', 'b_exo', 'Omega_x0', 'sigma8']

# Filter the plotter
g = plots.get_subplot_plotter()
g.triangle_plot(samples, params_to_keep, 
                filled=True, 
                title_limit=1,
                contour_colors=['#1f77b4'])

# If on a cluster, plt.show() might be trying to open a window that doesn't exist.
# ALWAYS export to a file to be safe.
g.export('my_triangle_plot.png')

In [6]:

# Cell 2: Hubble 1D Plot
# We define a NEW plotter to clear the buffer
g1d = plots.get_single_plotter(width_inch=5)
g1d.plot_1d(samples, 'H0', color='#1f77b4', lw=2)

plt.axvspan(66.82, 67.90, color='gray', alpha=0.15, label='Planck 2018')
plt.axvspan(72.0, 74.08, color='red', alpha=0.1, label='SH0ES (2022)')
plt.xlim(66,76)

plt.title(r'Hubble Constant Posterior ')
plt.xlabel(r'$H_0 \,\, [\mathrm{km/s/Mpc}]$')
plt.legend(loc='upper right')

# Explicitly save using the plotter's export
g1d.export('h0_comparison.png')
print("1D plot saved to h0_comparison.png")

1D plot saved to h0_comparison.png


In [8]:
# 1. Get Convergence Stats
r_minus_1 = samples.getGelmanRubin()
# getGelmanRubinEigenvalues returns the R-1 for each eigenvalue; 
# the first one is usually the "worst" one reported in logs.
r_minus_1_worst = samples.getGelmanRubinEigenvalues()[0] 

print("="*55)
print(f"{'MCMC CONVERGENCE DIAGNOSTICS':^55}")
print("="*55)
print(f"{'Total Samples (Rows):':<25} {samples.numrows}")
print(f"{'Total Weight (Effective):':<25} {samples.norm:.2f}")
print(f"{'Gelman-Rubin R-1 (mean):':<25} {r_minus_1:.5f}")
print(f"{'Worst R-1 (eigenvalue):':<25} {r_minus_1_worst:.5f}")

# Check if it meets the paper-standard (0.02)
status = "CONVERGED ✅" if r_minus_1_worst < 0.02 else "NOT CONVERGED ❌"
print(f"{'Status:':<25} {status}")
print("-" * 55)

# 2. Get Parameter Marginals
stats = samples.getMargeStats()
print(f"{'Parameter':<15} {'Mean':<12} {'1-sigma Error':<18}")
print("-" * 55)

for name in ['H0', 'Omega_m', 'a_exo', 'b_exo', 'Omega_x0', 'sigma8', 'Omega_b', 'rs_drag']:
    p = stats.parWithName(name)
    if p:
        # p.err is the standard deviation (68% limit)
        print(f"{name:<15} {p.mean:<12.5f} ± {p.err:<18.5f}")
print("="*55)

             MCMC CONVERGENCE DIAGNOSTICS              
Total Samples (Rows):     27274
Total Weight (Effective): 81234.00
Gelman-Rubin R-1 (mean):  0.01135
Worst R-1 (eigenvalue):   0.00010
Status:                   CONVERGED ✅
-------------------------------------------------------
Parameter       Mean         1-sigma Error     
-------------------------------------------------------
H0              69.63517     ± 0.49513           
Omega_m         0.30919      ± 0.00804           
a_exo           -966.27610   ± 530.84189         
b_exo           234.26116    ± 660.40252         
Omega_x0        -0.00527     ± 0.00290           
sigma8          0.84559      ± 0.02075           
Omega_b         0.04661      ± 0.00069           
rs_drag         145.13216    ± 1.31205           
